# ② 검색 — 색인 만들기 · 검색
담당: 검색 · 코드: `rag/retrieval/indexing.py`, `rag/retrieval/retrieval.py`

01 전처리의 `artifacts/chunks.jsonl` 로 Dense(KURE-v1 임베딩 → FAISS)와 BM25(Kiwi 형태소) 색인을 만들어 `artifacts/index/` 에 저장합니다.
- 임베딩은 청크를 `embedding.max_seq_length`(2,048) 토큰까지 읽습니다 — 표 청크가 잘리지 않습니다.
- BM25 는 요구사항 코드(`SFR-001`)를 한 토큰으로, 부정어 '않' · '아니' 도 남기고, 사용자 사전 · 별칭(부가세 → 부가가치세, 지체 상금 → 지체상금)으로 질문과 문서의 낱말을 맞춥니다.

검색은 두 점수를 섞은 하이브리드입니다.
- 질문의 요구사항 코드가 그대로 든 청크를 맨 앞에 둡니다.
- 질문에 발주기관이 나오면 그 기관 문서 안에서만 찾습니다. 기관명은 발주 기관 목록 CSV(`paths.org_csv`)와 문서 표지에서 모읍니다.
- 결과는 개수가 아니라 토큰 예산(`retrieval.token_budget`)으로 자르고, 거의 같은 청크는 하나만 남깁니다.

`.env` 에 Langfuse 키가 있으면 검색할 때마다 Langfuse Cloud 에 `retrieve-context` 로 기록됩니다 (없어도 똑같이 동작).

In [ ]:
import pandas as pd

from rag.config import ARTIFACTS, load_config, read_jsonl
from rag.retrieval.indexing import build_index
from rag.retrieval.retrieval import HybridIndex

cfg = load_config()

## 1. 색인 만들기
`임베딩에서 잘린 청크` 가 0 이면 모든 청크가 끝까지 임베딩된 것입니다.

In [ ]:
chunks = read_jsonl(ARTIFACTS / "chunks.jsonl")
build_index(chunks, cfg, ARTIFACTS / "index")

## 2. 검색해 보기
`config.yaml` 의 `retrieval` (token_budget · dense_weight · candidate_k)을 바꿔 가며 결과를 확인하세요.

In [ ]:
index = HybridIndex(ARTIFACTS, cfg)
QUESTION = "고려대학교 차세대 포털 시스템 사업의 응답 시간 요구사항은?"
pd.DataFrame(index.search(QUESTION))[["score", "title", "page", "kind", "section", "body"]]

## 3. 기관 라우팅 점검
기관명이 하나도 연결되지 않은 문서입니다. 이 문서들은 질문에 기관명이 나와도 전체 문서에서 찾습니다.

In [ ]:
routed = set().union(*index.router.org_docs.values())
print(f"라우팅되는 문서 {len(routed)}/{len(index.router.titles)}")
pd.DataFrame({"미라우팅 문서": sorted(t for d, t in index.router.titles.items() if d not in routed)})